> 在线讲义：[第 36 章 · CNN 训练的数学：卷积的反向传播、转置卷积、Xavier / He 初始化、BatchNorm 的梯度、残差的梯度流](https://wikieden.github.io/math-and-eval-stats/lessons/36-cnn-training.html) · 本书《数学与实验统计》

# 第 36 章 · CNN 训练的数学：卷积的反向传播、转置卷积、Xavier / He 初始化、BatchNorm 的梯度、残差的梯度流

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/36-cnn-training.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 反向 | Tᵀg | `conv_backward(g)` | [1, 2, −1, −3, 0, 1] |
| 第二步 转置卷积 | 尺寸公式 | `tconv_size(16, 4, 2, 1)` | 32 |
| 第三步 初始化 | ½n·Var | `layer_gain`、`one_layer_check` | 1.0000、1.0011 |
| 第四步 BN 梯度 | 投影 | `batchnorm_grad(x, g)` | Σ = 0 |
| 第五步 残差 | 连乘 | `grad_products()` | 1.11e-5 vs 1.8153 |

例子：第 35 章的一维信号和核（反向传播 = 转置）；转置卷积的尺寸；单层的方差增益（Xavier / He）；
BatchNorm 的梯度性质；残差连接的梯度连乘。
里程碑 ME-M23 的“深层网络在 Xavier / He / 随机初始化下的激活方差图”留给自己做；这里只算一层。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
# 下面几段来自 ch35_convolution.py（第 35 章）
import numpy as np
X = np.array([3.0, 1.0, 4.0, 1.0, 5.0, 9.0])

K = np.array([1.0, 0.0, -1.0])

def toeplitz_matrix(n: int, k: np.ndarray) -> np.ndarray:
    """valid 互相关写成矩阵：第 i 行在第 i..i+len(k)−1 列放核，其余是 0。"""
    m = n - len(k) + 1
    t = np.zeros((m, n))
    for i in range(m):
        t[i, i:i + len(k)] = k
    return t

**函数 `conv_backward`**：前向 y = T x（valid 互相关），反向 ∂L/∂x = Tᵀ g = g 和核的 full 卷积。

In [ ]:
def conv_backward(g: np.ndarray, k: np.ndarray = K) -> np.ndarray:
    """前向 y = T x（valid 互相关），反向 ∂L/∂x = Tᵀ g = g 和核的 full 卷积。"""
    return np.convolve(g, k, "full")

**函数 `tconv_size`**：转置卷积输出尺寸 (n − 1)s − 2p + d(k − 1) + output_padding + 1。

In [ ]:
def tconv_size(n: int, k: int, s: int = 1, p: int = 0, d: int = 1, op: int = 0) -> int:
    """转置卷积输出尺寸 (n − 1)s − 2p + d(k − 1) + output_padding + 1。"""
    return (n - 1) * s - 2 * p + d * (k - 1) + op + 1

**函数 `layer_gain`**：一层 y = Wx：E[y²] = n_in·Var(w)·E[x²]；过 ReLU 后二阶矩再减半。返回 E[a_out²]/E[a_in²]。

In [ ]:
def layer_gain(var_w: float, n_in: int, relu: bool = True) -> float:
    """一层 y = Wx：E[y²] = n_in·Var(w)·E[x²]；过 ReLU 后二阶矩再减半。返回 E[a_out²]/E[a_in²]。"""
    return n_in * var_w * (0.5 if relu else 1.0)

**函数 `one_layer_check`**：模拟一层：输入二阶矩 1，看输出 ReLU 后的二阶矩。

In [ ]:
def one_layer_check(var_w: float, n_in: int = 512, n: int = 4000, seed: int = 0) -> float:
    """模拟一层：输入二阶矩 1，看输出 ReLU 后的二阶矩。"""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, n_in))
    w = rng.normal(0, np.sqrt(var_w), (n_in, n_in))
    a = np.maximum(x @ w, 0)
    return float((a ** 2).mean())

**函数 `batchnorm_grad`**：BatchNorm（γ = 1、β = 0）对输入的梯度：dx = (g − mean(g) − x̂·mean(g·x̂)) / σ。

In [ ]:
def batchnorm_grad(x: np.ndarray, g: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """BatchNorm（γ = 1、β = 0）对输入的梯度：dx = (g − mean(g) − x̂·mean(g·x̂)) / σ。"""
    mu, var = x.mean(0), x.var(0)
    xh = (x - mu) / np.sqrt(var + eps)
    return (g - g.mean(0) - xh * (g * xh).mean(0)) / np.sqrt(var + eps)

**函数 `grad_products`**：普通堆叠：每层导数 w ~ U(0.5, 1.1)，连乘；残差：每层 1 + 0.1·u，u ~ U(−1, 1)。返回两个连乘。

In [ ]:
def grad_products(depth: int = 50, seed: int = 0) -> tuple[float, float]:
    """普通堆叠：每层导数 w ~ U(0.5, 1.1)，连乘；残差：每层 1 + 0.1·u，u ~ U(−1, 1)。返回两个连乘。"""
    rng = np.random.default_rng(seed)
    plain = float(np.prod(rng.uniform(0.5, 1.1, depth)))
    res = float(np.prod(1 + 0.1 * rng.uniform(-1, 1, depth)))
    return plain, res

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 卷积的反向传播 = 乘转置矩阵 = 和核做 full 卷积")
t = toeplitz_matrix(len(X), K)
g = np.array([1.0, 2.0, 0.0, -1.0])
print(f"  上游梯度 g = {g}：Tᵀg = {t.T @ g}；np.convolve(g, k, 'full') = {conv_backward(g)}")

In [ ]:
print("\n第二步 转置卷积：把小图放大")
for args in ((16, 4, 2, 1), (7, 3, 2, 1), (7, 3, 2, 1, 1, 1)):
    print(f"  输入 {args[0]}、核 {args[1]}、步长 {args[2]}、填充 {args[3]}{'、output_padding 1' if len(args) > 4 else ''}：输出 {tconv_size(*args)}")

In [ ]:
print("\n第三步 初始化：一层的方差增益")
n_in = 512
for name, vw in (("N(0, 0.01²)", 1e-4), ("Xavier 2/(n_in + n_out)", 2 / (2 * n_in)), ("He 2/n_in", 2 / n_in)):
    print(f"  {name}：理论增益 {layer_gain(vw, n_in):.4f}，模拟一层 ReLU 后二阶矩 {one_layer_check(vw):.4f}")

In [ ]:
print("\n第四步 BatchNorm 的梯度：减掉均值方向和 x̂ 方向")
rng = np.random.default_rng(1)
x = rng.normal(3, 2, (32, 4))
gg = rng.standard_normal((32, 4))
dx = batchnorm_grad(x, gg, eps=0.0)
xh = (x - x.mean(0)) / x.std(0)
print(f"  每个通道 Σ dx = {(np.round(dx.sum(0), 10) + 0.0).tolist()}；Σ dx·x̂ = {(np.round((dx * xh).sum(0), 10) + 0.0).tolist()}")

In [ ]:
print("\n第五步 残差：梯度连乘里多了一个 1")
p, r = grad_products()
print(f"  50 层，普通堆叠每层导数 U(0.5, 1.1)：连乘 {p:.2e}；残差每层 1 + 0.1·U(−1, 1)：连乘 {r:.4f}")

## 3. 代码里才看得到的

1. **核梯度没单独写函数**：∂L/∂k_j = Σ g_i x_{i+j}，就是 `np.correlate(x, g, "valid")`，留给题 1 自己验证。
2. **BN 检查时 eps = 0**：带 eps 时 x̂ 的分母和 `x.std()` 差一点，Σ dx·x̂ 就不是精确的 0。
3. **一层模拟用 4000 × 512 的随机输入**：二阶矩的估计误差约 1/√(4000·512)，所以和理论值差在千分位。

## 4. 自己改一改

1. 把 `one_layer_check` 的激活换成 tanh，Xavier 的增益变成多少？
2. 给 BN 加上 γ = 2、β = 1，梯度的两个正交性质还成立吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 卷积反向乘转置：对输入 full 卷积，对核互相关；
> 转置卷积拿来上采样，不是逆，注意棋盘格；
> 一层增益半个 n 乘方差，He 取二除 n；
> BN 梯度去掉均值和 x̂ 两个方向；
> 残差每层多个 1，连乘不消失。